<a href="https://colab.research.google.com/github/Luwijiiiiiiii/Mae-C-A-modified-Priority-Scheduling-System/blob/main/MaeC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# STEP 1
# PROCESS CLASS
# ============================================================

class Process:

    def __init__(self, pid, arrival, burst):

        self.pid = pid
        self.arrival = arrival
        self.burst = burst

        # Remaining burst time
        self.remaining = burst

        # Scheduling metrics
        self.start = None
        self.completion = None
        self.turnaround = None
        self.waiting = 0

        # MAE-C priority
        self.priority = 0

In [ ]:
# ============================================================
# STEP 2
# MAE-C SCHEDULER FUNCTION
# ============================================================

def maec_scheduler(processes):

    # Duplicate process list
    procs = [
        Process(p.pid, p.arrival, p.burst)
        for p in processes
    ]

    time = 0
    completed = 0
    n = len(procs)

    gantt = []
    current_pid = None

    # ========================================================
    # AUTOMATED AGING FACTOR
    # ========================================================

    max_burst = max(p.burst for p in procs)

    aging_factor = 1 / max_burst

    print("\n================================================")
    print("MAE-C AUTOMATED AGING CONFIGURATION")
    print("================================================")

    print(f"Maximum Burst Time : {max_burst}")
    print(f"Aging Factor       : {aging_factor:.5f}")

    # ========================================================
    # MAIN LOOP
    # ========================================================

    while completed < n:

        # ----------------------------------------------------
        # GET READY PROCESSES
        # ----------------------------------------------------

        ready = [
            p for p in procs
            if p.arrival <= time and p.remaining > 0
        ]

        # CPU idle
        if not ready:
            time += 1
            continue

        # ----------------------------------------------------
        # UPDATE WAITING TIME
        # ----------------------------------------------------

        for p in ready:

            p.waiting = (
                time
                - p.arrival
                - (p.burst - p.remaining)
            )

            if p.waiting < 0:
                p.waiting = 0

        # ----------------------------------------------------
        # COMPUTE MAE-C PRIORITY
        # ----------------------------------------------------

        for p in ready:

            p.priority = (
                p.burst
                - (p.waiting * aging_factor)
            )

        # ----------------------------------------------------
        # SELECT PROCESS
        # LOWEST VALUE = HIGHEST PRIORITY
        # ----------------------------------------------------

        current_process = min(
            ready,
            key=lambda x: x.priority
        )

        # ----------------------------------------------------
        # GANTT CHART CONTEXT SWITCH
        # ----------------------------------------------------

        if current_pid != current_process.pid:

            gantt.append(
                (current_process.pid, time)
            )

            current_pid = current_process.pid

        # ----------------------------------------------------
        # EXECUTE 1 TIME UNIT
        # ----------------------------------------------------

        if current_process.start is None:
            current_process.start = time

        current_process.remaining -= 1

        time += 1

        # ----------------------------------------------------
        # PROCESS FINISHED
        # ----------------------------------------------------

        if current_process.remaining == 0:

            current_process.completion = time

            current_process.turnaround = (
                current_process.completion
                - current_process.arrival
            )

            current_process.waiting = (
                current_process.turnaround
                - current_process.burst
            )

            completed += 1

    # ========================================================
    # FINALIZE GANTT CHART
    # ========================================================

    final_gantt = []

    for i in range(len(gantt)):

        pid, start = gantt[i]

        if i + 1 < len(gantt):
            end = gantt[i + 1][1]
        else:
            end = time

        final_gantt.append(
            (pid, start, end)
        )

    return procs, final_gantt, aging_factor

In [ ]:
# ============================================================
# STEP 3
# PRINT RESULTS
# ============================================================

def print_results(processes, gantt, aging_factor):

    print("\n================================================")
    print("MAE-C RESULTS")
    print("================================================")

    print(f"\nAutomated Aging Factor: {aging_factor:.5f}")

    # --------------------------------------------------------
    # GANTT CHART
    # --------------------------------------------------------

    print("\nGANTT CHART")
    print("------------------------------------------------")

    for g in gantt:

        print(f"{g[0]} : {g[1]} → {g[2]}")

    # --------------------------------------------------------
    # PROCESS TABLE
    # --------------------------------------------------------

    print("\nPROCESS TABLE")
    print("------------------------------------------------")

    print(
        "PID | AT | BT | CT | TAT | WT | PRIORITY"
    )

    for p in processes:

        print(
            f"{p.pid:>3} | "
            f"{p.arrival:>2} | "
            f"{p.burst:>2} | "
            f"{p.completion:>3} | "
            f"{p.turnaround:>3} | "
            f"{p.waiting:>3} | "
            f"{p.priority:.3f}"
        )

    # --------------------------------------------------------
    # AVERAGES
    # --------------------------------------------------------

    avg_tat = (
        sum(p.turnaround for p in processes)
        / len(processes)
    )

    avg_wt = (
        sum(p.waiting for p in processes)
        / len(processes)
    )

    print("\n================================================")
    print("AVERAGE METRICS")
    print("================================================")

    print(f"Average Turnaround Time : {avg_tat:.2f}")
    print(f"Average Waiting Time    : {avg_wt:.2f}")

In [ ]:
# ============================================================
# STEP 4
# USER INPUT
# ============================================================

processes = []

n = int(input("Enter number of processes: "))

for i in range(1, n + 1):

    print(f"\nPROCESS P{i}")

    arrival = int(
        input("Arrival Time : ")
    )

    burst = int(
        input("Burst Time   : ")
    )

    processes.append(
        Process(
            f"P{i}",
            arrival,
            burst
        )
    )

Enter number of processes: 3

PROCESS P1
Arrival Time : 2
Burst Time   : 5

PROCESS P2
Arrival Time : 56
Burst Time   : 1

PROCESS P3
Arrival Time : 1
Burst Time   : 89


In [ ]:
# ============================================================
# STEP 5
# RUN MAE-C
# ============================================================

results, gantt, aging_factor = (
    maec_scheduler(processes)
)

print_results(
    results,
    gantt,
    aging_factor
)


MAE-C AUTOMATED AGING CONFIGURATION
Maximum Burst Time : 89
Aging Factor       : 0.01124

MAE-C RESULTS

Automated Aging Factor: 0.01124

GANTT CHART
------------------------------------------------
P3 : 1 → 2
P1 : 2 → 7
P3 : 7 → 56
P2 : 56 → 57
P3 : 57 → 96

PROCESS TABLE
------------------------------------------------
PID | AT | BT | CT | TAT | WT | PRIORITY
 P1 |  2 |  5 |   7 |   5 |   0 | 5.000
 P2 | 56 |  1 |  57 |   1 |   0 | 1.000
 P3 |  1 | 89 |  96 |  95 |   6 | 88.933

AVERAGE METRICS
Average Turnaround Time : 33.67
Average Waiting Time    : 2.00
